# Primary And Business Ad Hoc Analysis of SQL in python

In [1]:
import mysql.connector
 

In [2]:
import sys
print(sys.executable)



C:\ProgramData\anaconda3\python.exe


In [3]:
 import sys
!{sys.executable} -m pip install mysql-connector-python pandas --quiet


In [4]:
 import pandas as pd

 

In [5]:
 conn = mysql.connector.connect(
    host='localhost',
    user='pythonuser',
    password='python1234',
    database='news',
    allow_local_infile=True
)

print("✅ Connected successfully!")


✅ Connected successfully!


In [12]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning)


In [15]:
from IPython.display import display, HTML

display(HTML("""
<style>
h1 {font-size: 30px !important;}
h2 {font-size: 24px !important;}
h3 {font-size: 18px !important;}
p, li {font-size: 20px !important;}
</style>
"""))


#

## total_circulation trends

In [16]:
query = """SELECT
    dc.city,
    DATE_FORMAT(dps.Month_numeric, '%Y-%m') AS month,
    SUM(dps.copies_sold) AS total_printed,
    SUM(dps.copies_sold) AS total_sold,
    SUM(dps.Net_Circulation) AS total_circulation
FROM fact_print_sales dps
JOIN dim_city dc ON dps.City_ID = dc.City_ID
WHERE dps.Month_numeric BETWEEN '2019-01-01' AND '2024-12-31'
GROUP BY dc.city, month
ORDER BY month, dc.city
"""
 

In [17]:
df = pd.read_sql(query, conn)
display(df)

,city,month,total_printed,total_sold,total_circulation
0,Ahmedabad,2019-01,324890.0,324890.0,311897.0
1,bhopal,2019-01,282436.0,282436.0,271668.0
2,Delhi,2019-01,396250.0,396250.0,380681.0
3,jaipur,2019-01,476937.0,476937.0,451463.0
4,kanpur,2019-01,362751.0,362751.0,348222.0
...,...,...,...,...,...
645,kanpur,2024-12,285163.0,285163.0,274059.0
646,Mumbai,2024-12,311533.0,311533.0,289465.0
647,Patna,2024-12,206311.0,206311.0,195436.0
648,ranchi,2024-12,188572.0,188572.0,179784.0


## top performing cities 2024


In [18]:
 query_top_cities = """
SELECT
    dc.city,
    SUM(dps.copies_sold) AS total_sold,
    SUM(dps.Net_Circulation) AS total_circulation
FROM fact_print_sales dps
JOIN dim_city dc ON dps.City_ID = dc.City_ID
WHERE YEAR(dps.Month_numeric) = 2024
GROUP BY dc.city
ORDER BY total_circulation DESC, total_sold DESC
LIMIT 10;
"""

 

In [20]:
 df_top_cities = pd.read_sql(query_top_cities, conn)
 display(df_top_cities)

,city,total_sold,total_circulation
0,jaipur,4361397.0,4128641.0
1,Varanasi,4357583.0,4123611.0
2,Mumbai,3775800.0,3569229.0
3,kanpur,3159143.0,2985718.0
4,Delhi,2912054.0,2726917.0
5,Ahmedabad,2649078.0,2518120.0
6,bhopal,2352492.0,2214277.0
7,Patna,2181912.0,2062729.0
8,ranchi,2018167.0,1919038.0
9,lucknow,1714702.0,1618183.0


## Waste analysis

In [21]:
query_waste = """
SELECT
    dc.city,
    DATE_FORMAT(dps.Month_numeric, '%Y-%m') AS month,
    SUM(dps.copies_sold) - SUM(dps.Net_Circulation) AS waste
FROM fact_print_sales dps
JOIN dim_city dc ON dps.City_ID = dc.City_ID
WHERE dps.Month_numeric BETWEEN '2019-01-01' AND '2024-12-31'
GROUP BY dc.city, month
ORDER BY waste DESC
"""
 

In [22]:
df_waste = pd.read_sql(query_waste, conn)


In [23]:
display(df_waste)

,city,month,waste
0,Varanasi,2019-12,38021.0
1,Varanasi,2019-01,37574.0
2,Varanasi,2019-04,36995.0
3,Varanasi,2020-12,33059.0
4,Varanasi,2019-11,33019.0
...,...,...,...
645,ranchi,2023-06,6080.0
646,lucknow,2022-04,5975.0
647,lucknow,2023-09,5466.0
648,lucknow,2022-02,5323.0


## Ad Revenue trends by category 

In [24]:
query_ad_revenue = """
 SELECT
    dac.standard_ad_category,
    SUBSTRING(quarter, 1, 4) AS year,
    SUM(ad_revenue) AS total_revenue,
    currency
FROM fact_ad_revenue far
join dim_ad_category dac on dac.ad_categoryid=far.ad_category
WHERE SUBSTRING(quarter, 1, 4) BETWEEN '2019' AND '2024'
GROUP BY dac.standard_ad_category, year, currency
ORDER BY year, dac.standard_ad_category, currency;
"""
 

In [25]:
df_ad_revenue = pd.read_sql(query_ad_revenue, conn)
display(df_ad_revenue)

,standard_ad_category,year,total_revenue,currency
0,Automobile,2019,8217028.00,IN RUPEES
1,Automobile,2019,9661628.00,INR
2,FMCG,2019,18776.23,EUR
3,FMCG,2019,17607138.00,INR
4,Government,2019,84909.10,EUR
...,...,...,...,...
62,Government,2024,116974.41,EUR
63,Government,2024,1465133.00,IN RUPEES
64,Government,2024,16473485.00,INR
65,Real Estate,2024,19243481.00,INR


## City level ad revenue performance

In [26]:
query_city_ad = """
SELECT
    dc.city,
    SUM(fas.ad_revenue) AS total_ad_revenue,
    SUM(dps.Net_Circulation) AS total_circulation
FROM fact_ad_revenue fas
JOIN fact_print_sales dps ON fas.edition_id = dps.edition_id
JOIN dim_city dc ON dps.City_ID = dc.City_ID
WHERE SUBSTRING(fas.quarter, 1, 4) BETWEEN '2019' AND '2024'
GROUP BY dc.city
ORDER BY total_ad_revenue DESC
"""
 

In [28]:
df_city_ad = pd.read_sql(query_city_ad, conn)
display(df_city_ad)

,city,total_ad_revenue,total_circulation
0,lucknow,3.257295e+09,199932228.0
1,bhopal,3.013358e+09,280236618.0
2,Ahmedabad,2.946980e+09,305145270.0
3,Mumbai,2.683489e+09,420155046.0
4,Patna,2.586907e+09,256611096.0
5,Delhi,2.531156e+09,370432044.0
6,ranchi,2.174533e+09,234888444.0
7,jaipur,2.081011e+09,469959372.0
8,Varanasi,1.890337e+09,496194948.0
9,kanpur,1.862676e+09,362365740.0


## Digital readiness by city

In [29]:
query_digital = """
SELECT
    dc.city,
    AVG(fcr.smartphone_penetration) AS avg_smartphone,
    AVG(fcr.internet_penetration) AS avg_internet,
    AVG(fcr.literacy_rate) AS avg_literacy
FROM fact_city_readiness fcr
JOIN dim_city dc ON fcr.city_id = dc.City_ID
WHERE SUBSTRING(fcr.quarter, 1, 4) BETWEEN '2019' AND '2024'
GROUP BY dc.city
ORDER BY avg_smartphone DESC, avg_internet DESC
"""
 

In [30]:
df_digital = pd.read_sql(query_digital, conn)
display(df_digital)

,city,avg_smartphone,avg_internet,avg_literacy
0,kanpur,78.842500,75.139583,71.314167
1,Varanasi,77.094583,74.730417,70.711250
2,ranchi,76.943750,62.522917,66.449583
3,lucknow,75.023750,56.388333,89.071250
4,bhopal,70.606250,66.530000,82.727083
5,jaipur,70.215417,10.000000,84.834167
6,Ahmedabad,68.762917,74.335833,75.106250
7,Patna,68.295000,67.648750,75.636667
8,Delhi,48.647917,48.882083,70.611250
9,Mumbai,48.481667,74.599167,81.894167


## Ad Revenue per Net Circulation by City and Year


In [31]:
query_roi = """
SELECT
    dc.city,
    SUBSTRING(fas.quarter, 1, 4) AS year,
    SUM(fas.ad_revenue) AS total_ad_revenue,
    SUM(dps.Net_Circulation) AS total_circulation,
    ROUND(SUM(fas.ad_revenue) / SUM(dps.Net_Circulation), 2) AS revenue_per_copy
FROM fact_ad_revenue fas
JOIN fact_print_sales dps ON fas.edition_id = dps.edition_id
JOIN dim_city dc ON dps.City_ID = dc.City_ID
WHERE SUBSTRING(fas.quarter, 1, 4) BETWEEN '2019' AND '2024'
AND dps.Net_Circulation > 0
GROUP BY dc.city, year
HAVING total_circulation > 0
ORDER BY revenue_per_copy DESC
"""

In [44]:
df_roi = pd.read_sql(query_roi, conn)
display(df_roi)


,city,year,total_ad_revenue,total_circulation,revenue_per_copy
0,lucknow,2024,6.097223e+08,33322038.0,18.30
1,lucknow,2021,5.860146e+08,33322038.0,17.59
2,Patna,2023,7.446555e+08,42768516.0,17.41
3,lucknow,2020,5.750270e+08,33322038.0,17.26
4,bhopal,2022,8.029096e+08,46706103.0,17.19
5,Ahmedabad,2020,8.698874e+08,50857545.0,17.10
6,lucknow,2019,5.624432e+08,33322038.0,16.88
7,lucknow,2023,5.308352e+08,33322038.0,15.93
8,ranchi,2020,5.387119e+08,39148074.0,13.76
9,bhopal,2019,6.427715e+08,46706103.0,13.76


##Digital Relaunch

In [34]:
query_prioritization = """
 SELECT
    dc.city,
    AVG(fcr.smartphone_penetration) AS avg_smartphone,
    AVG(fcr.internet_penetration) AS avg_internet,
    AVG(fcr.literacy_rate) AS avg_literacy,
    d2019.circulation_2019,
    d2024.circulation_2024
FROM dim_city dc
JOIN fact_city_readiness fcr ON fcr.city_id = dc.City_ID
LEFT JOIN (
    SELECT City_ID, SUM(Net_Circulation) AS circulation_2019
    FROM fact_print_sales
    WHERE YEAR(Month_numeric) = 2019
    GROUP BY City_ID
) d2019 ON d2019.City_ID = dc.City_ID
LEFT JOIN (
    SELECT City_ID, SUM(Net_Circulation) AS circulation_2024
    FROM fact_print_sales
    WHERE YEAR(Month_numeric) = 2024
    GROUP BY City_ID
) d2024 ON d2024.City_ID = dc.City_ID
WHERE SUBSTRING(fcr.quarter, 1, 4) BETWEEN '2019' AND '2024'
GROUP BY dc.city, dc.City_ID, d2019.circulation_2019, d2024.circulation_2024
HAVING circulation_2019 > 0 AND circulation_2024 > 0;
"""
 

In [35]:
df_priority = pd.read_sql(query_prioritization, conn)
display(df_priority)

,city,avg_smartphone,avg_internet,avg_literacy,circulation_2019,circulation_2024
0,lucknow,75.023750,56.388333,89.071250,2141504.0,1618183.0
1,Delhi,48.647917,48.882083,70.611250,3624575.0,2726917.0
2,bhopal,70.606250,66.530000,82.727083,2986344.0,2214277.0
3,Patna,68.295000,67.648750,75.636667,2769395.0,2062729.0
4,jaipur,70.215417,10.000000,84.834167,4640188.0,4128641.0
5,Mumbai,48.481667,74.599167,81.894167,4742773.0,3569229.0
6,ranchi,76.943750,62.522917,66.449583,2309184.0,1919038.0
7,kanpur,78.842500,75.139583,71.314167,3261935.0,2985718.0
8,Ahmedabad,68.762917,74.335833,75.106250,3324982.0,2518120.0
9,Varanasi,77.094583,74.730417,70.711250,5085718.0,4123611.0


# Ad Hoc Analysis

## ad hoc 1 - monthly circulation drop check

In [45]:
circulationdrop = """SELECT city, month, net_circulation, prev_circulation, (prev_circulation - net_circulation) AS drop_val  
FROM (
  SELECT
    dc.city,                                                           /* highest month over month drop*/
    DATE_FORMAT(fps.Month_numeric, '%Y-%m') AS month,
    SUM(fps.Net_Circulation) AS net_circulation,
    LAG(SUM(fps.Net_Circulation)) OVER (PARTITION BY dc.city ORDER BY fps.Month_numeric) AS prev_circulation
  FROM fact_print_sales fps
  JOIN dim_city dc ON fps.City_ID = dc.City_ID
  WHERE fps.Month_numeric BETWEEN '2019-01-01' AND '2024-12-31'
  GROUP BY dc.city, fps.Month_numeric
) t   
WHERE prev_circulation IS NOT NULL
ORDER BY (prev_circulation - net_circulation) DESC
LIMIT 3;"""


In [49]:
df_drop = pd.read_sql(circulationdrop, conn)
display(df_drop)

,city,month,net_circulation,prev_circulation,drop_val
0,Varanasi,2021-01,382018.0,441825.0,59807.0
1,Varanasi,2019-11,431606.0,487255.0,55649.0
2,jaipur,2020-01,420680.0,475361.0,54681.0


## 2. Yearly revenue concentration by category 


In [50]:
 revenue = """SELECT
    year,
    category_name,
    category_revenue,
    total_revenue_year,
    ROUND((category_revenue / total_revenue_year) * 100, 2) AS pct_of_categoryrevenue_total
FROM (
    SELECT
        SUBSTRING(quarter, 1, 4) AS year,
        dac.standard_ad_category AS category_name,
        SUM(far.ad_revenue) AS category_revenue,
        SUM(SUM(far.ad_revenue)) OVER (PARTITION BY SUBSTRING(quarter, 1, 4)) AS total_revenue_year
    FROM fact_ad_revenue far
    JOIN dim_ad_category dac ON dac.ad_categoryid = far.ad_category
    GROUP BY year, dac.standard_ad_category
) t
WHERE (category_revenue / total_revenue_year) > 0.40
ORDER BY year,  pct_of_categoryrevenue_total DESC;
"""


In [51]:
df_rev = pd.read_sql(revenue, conn)
display(df_rev)

,year,category_name,category_revenue,total_revenue_year,pct_of_categoryrevenue_total
0,2021,Real Estate,26818086.27,64574523.09,41.53


## Ad hoc -3 Print efficiency leader board
 


In [52]:
print = """SELECT 
    dc.city AS city_name,
    SUM(CASE WHEN YEAR(fps.Month_numeric) = 2024 THEN fps.copies_sold ELSE 0 END) AS copies_printed_2024,
    SUM(CASE WHEN YEAR(fps.Month_numeric) = 2024 THEN fps.Net_Circulation ELSE 0 END) AS net_circulation_2024,
    CASE 
      WHEN SUM(CASE WHEN YEAR(fps.Month_numeric) = 2024 THEN fps.copies_sold ELSE 0 END) = 0 THEN 0
      ELSE ROUND(
        SUM(CASE WHEN YEAR(fps.Month_numeric) = 2024 THEN fps.Net_Circulation ELSE 0 END) /
        SUM(CASE WHEN YEAR(fps.Month_numeric) = 2024 THEN fps.copies_sold ELSE 0 END),
      4)
    END AS efficiency_ratio,
    RANK() OVER (ORDER BY 
      SUM(CASE WHEN YEAR(fps.Month_numeric) = 2024 THEN fps.Net_Circulation ELSE 0 END) /
      NULLIF(SUM(CASE WHEN YEAR(fps.Month_numeric) = 2024 THEN fps.copies_sold ELSE 0 END), 0) DESC) AS efficiency_rank_2024
FROM fact_print_sales fps
JOIN dim_city dc ON fps.City_ID = dc.City_ID
WHERE YEAR(fps.Month_numeric) = 2024
GROUP BY dc.city
ORDER BY efficiency_rank_2024
LIMIT 5;"""

In [53]:
df_pr = pd.read_sql(print, conn)
display(df_pr)

,city_name,copies_printed_2024,net_circulation_2024,efficiency_ratio,efficiency_rank_2024
0,ranchi,2018167.0,1919038.0,0.9509,1
1,Ahmedabad,2649078.0,2518120.0,0.9506,2
2,jaipur,4361397.0,4128641.0,0.9466,3
3,Varanasi,4357583.0,4123611.0,0.9463,4
4,Patna,2181912.0,2062729.0,0.9454,5


## 4. Internet readiness growth 2021


In [54]:
 ready = """  WITH internet_rates AS (
    SELECT 
        dc.city AS city_name,
        fcr.quarter,
        AVG(fcr.internet_penetration) AS internet_rate
    FROM fact_city_readiness fcr
    JOIN dim_city dc ON fcr.city_id = dc.City_ID
    WHERE fcr.quarter IN ('2021-Q1', '2021-Q2', '2021-Q3', '2021-Q4')
    GROUP BY dc.city, fcr.quarter
)
SELECT 
    city_name,
    MAX(CASE WHEN quarter = '2021-Q1' THEN internet_rate END) AS internet_rate_q1_2021,
    MAX(CASE WHEN quarter = '2021-Q2' THEN internet_rate END) AS internet_rate_q2_2021,
    MAX(CASE WHEN quarter = '2021-Q3' THEN internet_rate END) AS internet_rate_q3_2021,
    MAX(CASE WHEN quarter = '2021-Q4' THEN internet_rate END) AS internet_rate_q4_2021,
    MAX(CASE WHEN quarter = '2021-Q4' THEN internet_rate END) - MAX(CASE WHEN quarter = '2021-Q1' THEN internet_rate END) AS delta_internet_rate
FROM internet_rates
GROUP BY city_name
ORDER BY delta_internet_rate desc
LIMIT 3;

SELECT DISTINCT quarter 
FROM fact_city_readiness 
WHERE quarter LIKE '2021%'
ORDER BY quarter;"""

In [55]:
df_read = pd.read_sql(ready, conn)
display(df_read)

,city_name,internet_rate_q1_2021,internet_rate_q2_2021,internet_rate_q3_2021,internet_rate_q4_2021,delta_internet_rate
0,kanpur,74.27,75.98,74.33,76.77,2.50
1,Mumbai,73.31,75.12,76.05,75.74,2.43
2,Ahmedabad,73.03,74.62,75.27,74.80,1.77


## 5. Consistent Multi year decline


In [19]:
consistent = """ WITH yearly_print AS (
    SELECT
        dc.city AS city_name,
        YEAR(fps.Month_numeric) AS year,
        CAST(SUM(fps.Net_Circulation) AS SIGNED) AS net_circulation
    FROM fact_print_sales fps
    JOIN dim_city dc ON fps.City_ID = dc.City_ID
    WHERE fps.Month_numeric BETWEEN '2019-01-01' AND '2024-12-31'
    GROUP BY dc.city, year
),
yearly_ad_revenue AS (
    SELECT
        dc.city AS city_name,
        SUBSTRING(far.quarter, 1, 4) AS year,
        CAST(SUM(far.ad_revenue) AS SIGNED) AS ad_revenue
    FROM fact_ad_revenue far
    JOIN fact_print_sales fps ON far.edition_id = fps.edition_id
    JOIN dim_city dc ON fps.city_id = dc.city_id
    WHERE SUBSTRING(far.quarter, 1, 4) BETWEEN '2019' AND '2024'
    GROUP BY dc.city, year
),
combined AS (
    SELECT
        yp.city_name,
        yp.year,
        yp.net_circulation,
        COALESCE(ya.ad_revenue, 0) AS ad_revenue
    FROM yearly_print yp
    LEFT JOIN yearly_ad_revenue ya ON yp.city_name = ya.city_name AND yp.year = ya.year
),
lagged AS (
    SELECT
        city_name,
        year,
        net_circulation,
        ad_revenue,
        LAG(net_circulation) OVER (PARTITION BY city_name ORDER BY year) AS prev_net_circulation,
        LAG(ad_revenue) OVER (PARTITION BY city_name ORDER BY year) AS prev_ad_revenue
    FROM combined
)
SELECT
    city_name,
    year,
    net_circulation,
    ad_revenue,
    CASE WHEN prev_net_circulation IS NOT NULL AND net_circulation < prev_net_circulation THEN 'Yes' ELSE 'No' END AS net_circulation_decline,
    CASE WHEN prev_ad_revenue IS NOT NULL AND ad_revenue < prev_ad_revenue THEN 'Yes' ELSE 'No' END AS ad_revenue_decline
FROM lagged
ORDER BY city_name, year;
"""
        

In [20]:
df_dec = pd.read_sql(consistent, conn)
display(df_dec)

,city_name,year,net_circulation,ad_revenue,net_circulation_decline,ad_revenue_decline
0,Ahmedabad,2019,3324982,463854341,No,No
1,Ahmedabad,2020,2585663,869887360,Yes,No
2,Ahmedabad,2021,2752515,446825173,No,Yes
3,Ahmedabad,2022,3109889,423340800,No,Yes
4,Ahmedabad,2023,2661346,443651200,Yes,No
5,Ahmedabad,2024,2518120,299421329,Yes,Yes
6,bhopal,2019,2986344,642771492,No,No
7,bhopal,2020,2794823,269193421,Yes,Yes
8,bhopal,2021,2675078,432790395,Yes,No
9,bhopal,2022,2731598,802909602,No,No


## 6. readiness 2021 vs pilot engagement outlier 2021


In [10]:
  
 pilot = """ WITH readiness_2021 AS (
    SELECT
        dc.city AS city_name,
        AVG((smartphone_penetration + internet_penetration + literacy_rate) / 3) AS readiness_score
    FROM fact_city_readiness fcr
    JOIN dim_city dc ON fcr.city_id = dc.City_ID
    WHERE LEFT(fcr.quarter,4) = '2021' -- filter all quarters in 2021
    GROUP BY dc.city
),
engagement_2021 AS (
    SELECT
        dc.city AS city_name,
        AVG(fe.avg_bounce_rate) AS engagement_metric
    FROM fact_digital_pilot fe
    JOIN dim_city dc ON fe.city_id = dc.City_ID
    GROUP BY dc.city
),
ranked AS (
    SELECT
        r.city_name,
        r.readiness_score,
        e.engagement_metric,
        RANK() OVER (ORDER BY r.readiness_score DESC) AS readiness_rank_desc,
        RANK() OVER (ORDER BY e.engagement_metric asc) AS engagement_rank_asc
    FROM readiness_2021 r
    JOIN engagement_2021 e ON r.city_name = e.city_name
)
SELECT
    city_name,
    readiness_score AS readiness_score_2021,
    engagement_metric AS engagement_metric_2021,
    readiness_rank_desc,
    engagement_rank_asc,
    CASE
        WHEN readiness_rank_desc = 1 AND engagement_rank_asc <= 3 THEN 'Yes' ELSE 'No'
    END AS is_outlier
FROM ranked
ORDER BY readiness_rank_desc;
"""

In [13]:
df_pi = pd.read_sql(pilot, conn)
display(df_pi)

,city_name,readiness_score_2021,engagement_metric_2021,readiness_rank_desc,engagement_rank_asc,is_outlier
0,kanpur,75.230833,72.7980,1,9,No
1,Varanasi,73.887500,74.7775,2,10,No
2,bhopal,73.210000,61.8100,3,3,No
3,lucknow,73.204167,62.7840,4,4,No
4,Ahmedabad,72.393333,60.8525,5,2,No
5,Patna,70.770833,63.8540,6,6,No
6,ranchi,68.640833,67.8660,7,7,No
7,Mumbai,68.331667,63.4900,8,5,No
8,Delhi,56.075833,60.8080,9,1,No
9,jaipur,54.947500,69.4520,10,8,No
